In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/example_data_classification.csv?raw=true', header=None)

In [3]:
data

,0,1,2
0,34.623660,78.024693,0
1,30.286711,43.894998,0
2,35.847409,72.902198,0
3,60.182599,86.308552,1
4,79.032736,75.344376,1
...,...,...,...
95,83.489163,48.380286,1
96,42.261701,87.103851,1
97,99.315009,68.775409,1
98,55.340018,64.931938,1


In [41]:
# the goal is to code our own Logistic Regression
class LR():
    def __init__(self):
        self.w = None
        self.b = None
        
    def sig(self,z):
        return 1/(1+np.exp(-z))

    def loss(self,x,y):
        predictions = self.sig(x@self.w+self.b)
        return -np.mean(y*np.log(predictions)+(1-y)*np.log(1-predictions))
        
    def gradient(self,x,y):
        error = y - self.sig(x@self.w+self.b)
        return (-1/len(x))*error@x, (-1/len(x))*sum(error)
        
    def fit(self,x,y,lr=0.001,maxiter=1000,tol=1e-6):
        # we must initialize our weights
        self.w = np.random.normal(size=x.shape[1])
        self.b = np.random.normal()
        u = 0 
        for _ in range(maxiter):     
            gw, gb = self.gradient(x,y)
            self.wnew = self.w - lr*gw
            self.bnew = self.b - lr*gb
            u += 1

            if (u+1)%100==0:
                print(f'After {u+1} iterations the loss is {self.loss(x,y)}')
            # implement the tolerance check
            if np.linalg.norm(self.wnew-self.w)<tol:
                print(f'The algorithm has converged and the loss is {self.loss(x,y)}')
                break
            self.w = self.wnew
            self.b = self.bnew
    def predict(self,x,thresh=0.5):
        self.thresh = thresh
        return (self.sig(x@self.w+self.b)>self.thresh) + 0
    # here score means accuracy
    def score(self,x,y,thresh=0.5):
        return 1 - sum(abs(y-self.predict(x,thresh)))/len(y)

In [42]:
x = data.iloc[:,:-1].values
y = data.iloc[:,-1].values

In [43]:
model1 = LR()

In [46]:
model1.fit(x,y,lr=0.001,maxiter=500)

After 100 iterations the loss is 0.7048722384469968
After 200 iterations the loss is 0.6634645732714265
The algorithm has converged and the loss is 0.6634482091819579


In [47]:
model1.score(x,y,thresh=0.5)

np.float64(0.6)

In [9]:
from sklearn.linear_model import LogisticRegression

In [10]:
model2 = LogisticRegression()

In [11]:
model2.fit(x,y)
model2.score(x,y)

0.89

In [48]:
# RMSPROP approach
class LR_rmsprop:
    def __init__(self):
        self.w = None
        self.b = None

    def loss(self,x,y):
        predictions = self.sig(x@self.w+self.b)
        log_lik = np.mean(y * np.log(predictions) + (1 - y) * np.log(1 - predictions))
        return -log_lik

    def sig(self,z):
        return 1/(1+np.exp(-z))
        
    def gradient(self,x,y,w,b):
        errors = y - self.sig(x@w+b)
        return -1/len(x)*(errors)@x, -1/len(x)*sum(errors)
        
    def fit(self,x,y,lr=0.01,maxiter=1000,intercept=True,tol=1e-5,sw=0,sb=0,eps=1e-6,batch_size=32,done=False,beta1=0.9,u=0):
        # the main goal of this is to update the weights with gradient descent
        # first initialize the weights and the bias term
        self.w = np.random.normal(size=x.shape[1])
        if intercept:
            self.b = np.random.normal()
        else:
            self.b = 0
        lss= []
        # Root Mean Squared Propagation
        for i in range(maxiter):
            minibatches = np.array_split(np.random.permutation(range(len(x))),len(x)//batch_size)
            for batch in minibatches:
                gw, gb = self.gradient(x,y,self.w,self.b)
        
                # here we create adaptive learning rates
                sw = beta1*sw + (1-beta1)*sum(gw**2)
                sb = beta1*sb + (1-beta1)*gb**2
                
                self.wnew = self.w - lr/np.sqrt(sw+eps)*gw
                self.bnew = self.b - lr/np.sqrt(sb+eps)*gb
                u += 1
                lss.append(self.loss(x,y))
                if np.linalg.norm(self.wnew-self.w)<tol:
                    print('The Algorithm has Converged!')
                    done = True
                    break
                if (u+1)%100 ==0:
                    print(f'After {u+1} updates the Loss is: {self.loss(x,y)}')
                self.w = self.wnew
                self.b = self.bnew
            if done:
                break
            
    def predict_proba(self,x):
        return self.sig(x@self.w+self.b)

    def predict_classes(self,x,thresh=0.5):
        return (self.sig(x@self.w+self.b)>thresh) + 0

    def score(self,x,y):
        return 1 - sum(abs(y-self.predict_classes(x)))/len(y)

In [50]:
model3 = LR_rmsprop()
model3.fit(x,y,batch_size=16)
model3.score(x,y)


After 100 updates the Loss is: 1.3167432274594895
After 200 updates the Loss is: 0.5409768411026665
After 300 updates the Loss is: 0.5158049138341212
After 400 updates the Loss is: 0.4931627690927508
After 500 updates the Loss is: 0.47277218769326723
After 600 updates the Loss is: 0.45437684685084845
After 700 updates the Loss is: 0.4377481656274632
After 800 updates the Loss is: 0.42268374072527437
After 900 updates the Loss is: 0.40900523937507843
After 1000 updates the Loss is: 0.3965560559863833
After 1100 updates the Loss is: 0.38519893755761
After 1200 updates the Loss is: 0.37481370510363626
After 1300 updates the Loss is: 0.365295142581591
After 1400 updates the Loss is: 0.3565510868785797
After 1500 updates the Loss is: 0.3485007279783616
After 1600 updates the Loss is: 0.3410731135973788
After 1700 updates the Loss is: 0.3342058443231894
After 1800 updates the Loss is: 0.32784394133499484
After 1900 updates the Loss is: 0.32193886753200013
After 2000 updates the Loss is: 0.31

np.float64(0.92)